# Streaming Activity Exploration: EDA, AI, ML, and DL

This notebook performs comprehensive analysis of streaming activity data including:
- **Exploratory Data Analysis (EDA)**: Understanding patterns in listening behavior
- **Machine Learning (ML)**: Clustering and recommendation systems
- **Deep Learning (DL)**: Neural networks for preference prediction
- **AI Insights**: Pattern recognition and predictive analytics

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

In [ ]:
# Load datasets
streaming_df = pd.read_csv('My Streaming Activity.csv')
features_df = pd.read_csv('Scrobble_Features.csv')

print(f"Streaming Activity Shape: {streaming_df.shape}")
print(f"Features Shape: {features_df.shape}")
print("\nStreaming Activity Columns:")
print(streaming_df.columns.tolist())
print("\nFeatures Columns:")
print(features_df.columns.tolist())

## 2. Data Preprocessing

In [ ]:
# Convert timestamps
streaming_df['TimeStamp_UTC'] = pd.to_datetime(streaming_df['TimeStamp_UTC'])
streaming_df['TimeStamp_Central'] = pd.to_datetime(streaming_df['TimeStamp_Central'])

# Extract temporal features
streaming_df['Year'] = streaming_df['TimeStamp_UTC'].dt.year
streaming_df['Month'] = streaming_df['TimeStamp_UTC'].dt.month
streaming_df['Day'] = streaming_df['TimeStamp_UTC'].dt.day
streaming_df['Hour'] = streaming_df['TimeStamp_UTC'].dt.hour
streaming_df['DayOfWeek'] = streaming_df['TimeStamp_UTC'].dt.dayofweek
streaming_df['WeekOfYear'] = streaming_df['TimeStamp_UTC'].dt.isocalendar().week

print("Temporal features extracted successfully!")
streaming_df.head()

In [ ]:
# Merge datasets on common columns
merged_df = streaming_df.merge(
    features_df, 
    on=['Performer', 'Song'], 
    how='left',
    suffixes=('_streaming', '_features')
)

print(f"Merged dataset shape: {merged_df.shape}")
print(f"\nMissing values in key audio features:")
audio_features = ['danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness']
print(merged_df[audio_features].isnull().sum())

## 3. Exploratory Data Analysis (EDA)

### 3.1 Basic Statistics

In [ ]:
print("=== Streaming Activity Statistics ===")
print(f"Total songs played: {len(streaming_df)}")
print(f"Unique songs: {streaming_df['Song'].nunique()}")
print(f"Unique performers: {streaming_df['Performer'].nunique()}")
print(f"Unique albums: {streaming_df['Album'].nunique()}")
print(f"\nDate range: {streaming_df['TimeStamp_UTC'].min()} to {streaming_df['TimeStamp_UTC'].max()}")
print(f"Total days: {(streaming_df['TimeStamp_UTC'].max() - streaming_df['TimeStamp_UTC'].min()).days}")

### 3.2 Top Artists and Songs

In [ ]:
# Top 10 artists
top_artists = streaming_df['Performer'].value_counts().head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top artists bar plot
top_artists.plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_xlabel('Number of Plays')
ax1.set_title('Top 10 Most Played Artists')
ax1.invert_yaxis()

# Top songs
top_songs = streaming_df['Song'].value_counts().head(10)
top_songs.plot(kind='barh', ax=ax2, color='coral')
ax2.set_xlabel('Number of Plays')
ax2.set_title('Top 10 Most Played Songs')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

print("\nTop 5 Artists:")
print(top_artists.head())
print("\nTop 5 Songs:")
print(top_songs.head())

### 3.3 Temporal Analysis

In [ ]:
# Listening patterns by hour of day
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Hour of day
hourly_counts = streaming_df['Hour'].value_counts().sort_index()
axes[0, 0].bar(hourly_counts.index, hourly_counts.values, color='skyblue')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Number of Songs')
axes[0, 0].set_title('Listening Activity by Hour of Day')
axes[0, 0].set_xticks(range(0, 24))

# Day of week
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = streaming_df['DayOfWeek'].value_counts().sort_index()
axes[0, 1].bar(range(7), dow_counts.values, color='lightcoral')
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('Number of Songs')
axes[0, 1].set_title('Listening Activity by Day of Week')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(day_names, rotation=45)

# Monthly trend
streaming_df['YearMonth'] = streaming_df['TimeStamp_UTC'].dt.to_period('M')
monthly_counts = streaming_df.groupby('YearMonth').size()
axes[1, 0].plot(monthly_counts.index.astype(str), monthly_counts.values, marker='o', color='green')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Number of Songs')
axes[1, 0].set_title('Monthly Listening Trends')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].set_xticks(axes[1, 0].get_xticks()[::3])  # Show every 3rd label

# Songs per day distribution
daily_counts = streaming_df.groupby(streaming_df['TimeStamp_UTC'].dt.date).size()
axes[1, 1].hist(daily_counts.values, bins=30, color='purple', alpha=0.7)
axes[1, 1].set_xlabel('Songs per Day')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Daily Listening Activity')

plt.tight_layout()
plt.show()

### 3.4 Audio Features Analysis

In [ ]:
# Filter rows with audio features
features_available = merged_df.dropna(subset=audio_features)

print(f"Songs with audio features: {len(features_available)} out of {len(merged_df)}")
print(f"Coverage: {len(features_available)/len(merged_df)*100:.2f}%")

if len(features_available) > 0:
    # Distribution of audio features
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, feature in enumerate(audio_features):
        axes[idx].hist(features_available[feature], bins=30, color='teal', alpha=0.7)
        axes[idx].set_xlabel(feature.capitalize())
        axes[idx].set_ylabel('Frequency')
        axes[idx].set_title(f'Distribution of {feature.capitalize()}')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\nAudio Features Summary:")
    print(features_available[audio_features].describe())

In [ ]:
# Correlation between audio features
if len(features_available) > 0:
    plt.figure(figsize=(10, 8))
    correlation_matrix = features_available[audio_features].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Correlation Matrix of Audio Features')
    plt.tight_layout()
    plt.show()

## 4. Machine Learning Models

### 4.1 Clustering Analysis (K-Means)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Prepare data for clustering
if len(features_available) > 0:
    clustering_features = audio_features[:5]  # Use top 5 features
    X_cluster = features_available[clustering_features].values
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_cluster)
    
    # Determine optimal number of clusters using elbow method
    inertias = []
    k_range = range(2, 11)
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(X_scaled)
        inertias.append(kmeans.inertia_)
    
    # Plot elbow curve
    plt.figure(figsize=(10, 5))
    plt.plot(k_range, inertias, marker='o', linewidth=2)
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('Inertia')
    plt.title('Elbow Method for Optimal k')
    plt.grid(True)
    plt.show()

In [ ]:
# Perform clustering with optimal k (e.g., k=4)
if len(features_available) > 0:
    optimal_k = 4
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    features_available['Cluster'] = kmeans.fit_predict(X_scaled)
    
    # Visualize clusters using PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], 
                         c=features_available['Cluster'], 
                         cmap='viridis', alpha=0.6, s=50)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
    plt.title('Song Clusters based on Audio Features')
    plt.colorbar(scatter, label='Cluster')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Cluster characteristics
    print("\nCluster Characteristics:")
    for cluster in range(optimal_k):
        cluster_data = features_available[features_available['Cluster'] == cluster]
        print(f"\n--- Cluster {cluster} ({len(cluster_data)} songs) ---")
        print(cluster_data[clustering_features].mean())

### 4.2 Recommendation System (Collaborative Filtering)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_songs(song_name, top_n=5):
    """Recommend similar songs based on audio features"""
    if len(features_available) == 0:
        print("No audio features available for recommendations")
        return None
    
    # Find the song
    song_data = features_available[features_available['Song'] == song_name]
    
    if len(song_data) == 0:
        print(f"Song '{song_name}' not found in dataset")
        return None
    
    # Get features for the song
    song_features = song_data[audio_features].values
    
    # Calculate similarity with all songs
    all_features = features_available[audio_features].values
    similarities = cosine_similarity(song_features, all_features)[0]
    
    # Get top N similar songs (excluding the query song itself)
    similar_indices = similarities.argsort()[::-1][1:top_n+1]
    
    recommendations = features_available.iloc[similar_indices][['Song', 'Performer']]
    recommendations['Similarity'] = similarities[similar_indices]
    
    return recommendations

# Test recommendation system
if len(features_available) > 0:
    test_song = features_available['Song'].iloc[0]
    print(f"Songs similar to '{test_song}':")
    recommendations = recommend_songs(test_song)
    if recommendations is not None:
        print(recommendations)

### 4.3 Play Count Prediction (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Create a feature for song popularity (play count)
song_popularity = streaming_df.groupby('Song').size().reset_index(name='PlayCount')

# Merge with features
ml_data = features_df.merge(song_popularity, on='Song', how='inner')
ml_data = ml_data.dropna(subset=audio_features + ['PlayCount'])

if len(ml_data) > 100:  # Need sufficient data
    X = ml_data[audio_features].values
    y = ml_data['PlayCount'].values
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train Random Forest model
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    
    # Predictions
    y_pred = rf_model.predict(X_test)
    
    # Evaluation
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"Random Forest Model Performance:")
    print(f"Mean Absolute Error: {mae:.2f}")
    print(f"R² Score: {r2:.3f}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'Feature': audio_features,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='forestgreen')
    plt.xlabel('Importance')
    plt.title('Feature Importance for Play Count Prediction')
    plt.tight_layout()
    plt.show()
    
    print("\nFeature Importance:")
    print(feature_importance)
else:
    print("Insufficient data for ML model training")

## 5. Deep Learning Models

### 5.1 Neural Network for Song Preference Prediction

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler

# Prepare data for deep learning
if len(ml_data) > 100:
    # Create binary target: high preference (top 25% play count)
    threshold = ml_data['PlayCount'].quantile(0.75)
    ml_data['HighPreference'] = (ml_data['PlayCount'] >= threshold).astype(int)
    
    X_dl = ml_data[audio_features].values
    y_dl = ml_data['HighPreference'].values
    
    # Normalize features
    scaler_dl = MinMaxScaler()
    X_dl_scaled = scaler_dl.fit_transform(X_dl)
    
    # Split data
    X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
        X_dl_scaled, y_dl, test_size=0.2, random_state=42
    )
    
    print(f"Training set size: {len(X_train_dl)}")
    print(f"Test set size: {len(X_test_dl)}")
    print(f"Positive class proportion: {y_dl.mean():.2%}")

In [ ]:
# Build neural network model
if len(ml_data) > 100:
    model = models.Sequential([
        layers.Input(shape=(len(audio_features),)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Compile model
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    
    print("Neural Network Architecture:")
    model.summary()

In [ ]:
# Train the model
if len(ml_data) > 100:
    history = model.fit(
        X_train_dl, y_train_dl,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        verbose=0,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
        ]
    )
    
    # Evaluate on test set
    test_loss, test_accuracy, test_auc = model.evaluate(X_test_dl, y_test_dl, verbose=0)
    
    print(f"\nTest Performance:")
    print(f"Loss: {test_loss:.4f}")
    print(f"Accuracy: {test_accuracy:.4f}")
    print(f"AUC: {test_auc:.4f}")
    
    # Plot training history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.legend()
    ax1.grid(True)
    
    # Loss
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.set_title('Model Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

### 5.2 Autoencoder for Feature Learning

In [ ]:
# Build autoencoder for dimensionality reduction and feature learning
if len(features_available) > 100:
    X_autoencoder = features_available[audio_features].values
    scaler_ae = StandardScaler()
    X_ae_scaled = scaler_ae.fit_transform(X_autoencoder)
    
    encoding_dim = 3  # Compressed representation dimension
    
    # Encoder
    encoder_input = layers.Input(shape=(len(audio_features),))
    encoded = layers.Dense(32, activation='relu')(encoder_input)
    encoded = layers.Dense(16, activation='relu')(encoded)
    encoded = layers.Dense(encoding_dim, activation='relu')(encoded)
    
    # Decoder
    decoded = layers.Dense(16, activation='relu')(encoded)
    decoded = layers.Dense(32, activation='relu')(decoded)
    decoded = layers.Dense(len(audio_features), activation='linear')(decoded)
    
    # Autoencoder model
    autoencoder = models.Model(encoder_input, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    
    # Train
    autoencoder.fit(
        X_ae_scaled, X_ae_scaled,
        epochs=50,
        batch_size=64,
        shuffle=True,
        validation_split=0.2,
        verbose=0
    )
    
    # Extract encoder for visualization
    encoder_model = models.Model(encoder_input, encoded)
    encoded_features = encoder_model.predict(X_ae_scaled)
    
    # Visualize encoded features
    if encoding_dim >= 3:
        from mpl_toolkits.mplot3d import Axes3D
        
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(111, projection='3d')
        scatter = ax.scatter(encoded_features[:, 0], 
                           encoded_features[:, 1], 
                           encoded_features[:, 2],
                           c=features_available['Cluster'] if 'Cluster' in features_available.columns else 'blue',
                           cmap='viridis', alpha=0.6)
        ax.set_xlabel('Encoded Feature 1')
        ax.set_ylabel('Encoded Feature 2')
        ax.set_zlabel('Encoded Feature 3')
        ax.set_title('Autoencoder Learned Features')
        plt.colorbar(scatter)
        plt.show()
    
    print("Autoencoder trained successfully!")
    print(f"Original dimension: {len(audio_features)}")
    print(f"Encoded dimension: {encoding_dim}")

## 6. AI Insights and Pattern Recognition

### 6.1 Listening Behavior Patterns

In [ ]:
# Analyze listening patterns by time of day and day of week
listening_patterns = streaming_df.pivot_table(
    values='index', 
    index='Hour', 
    columns='DayOfWeek', 
    aggfunc='count'
)

plt.figure(figsize=(12, 8))
sns.heatmap(listening_patterns, cmap='YlOrRd', annot=False, fmt='d', cbar_kws={'label': 'Song Count'})
plt.xlabel('Day of Week')
plt.ylabel('Hour of Day')
plt.title('Listening Activity Heatmap: Hour vs Day of Week')
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
plt.xticks(range(7), day_labels)
plt.tight_layout()
plt.show()

# Identify peak listening times
peak_hour = streaming_df['Hour'].mode()[0]
peak_day = streaming_df['DayOfWeek'].mode()[0]
print(f"\nPeak listening hour: {peak_hour}:00")
print(f"Peak listening day: {day_names[peak_day]}")

### 6.2 Music Taste Evolution

In [ ]:
# Analyze how audio feature preferences change over time
if len(features_available) > 100:
    features_available['Date'] = pd.to_datetime(features_available['TimeStamp_UTC']).dt.date
    
    # Calculate monthly average features
    features_available['YearMonth'] = pd.to_datetime(features_available['TimeStamp_UTC']).dt.to_period('M')
    monthly_features = features_available.groupby('YearMonth')[audio_features[:4]].mean()
    
    # Plot trends
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, feature in enumerate(audio_features[:4]):
        axes[idx].plot(monthly_features.index.astype(str), monthly_features[feature], 
                      marker='o', linewidth=2)
        axes[idx].set_xlabel('Month')
        axes[idx].set_ylabel(feature.capitalize())
        axes[idx].set_title(f'Evolution of {feature.capitalize()} Preference')
        axes[idx].tick_params(axis='x', rotation=45)
        axes[idx].set_xticks(axes[idx].get_xticks()[::3])
        axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Summary and Key Findings

In [ ]:
print("="*60)
print("STREAMING ACTIVITY ANALYSIS SUMMARY")
print("="*60)

print(f"\n📊 Dataset Overview:")
print(f"  • Total songs played: {len(streaming_df):,}")
print(f"  • Unique songs: {streaming_df['Song'].nunique():,}")
print(f"  • Unique artists: {streaming_df['Performer'].nunique():,}")
print(f"  • Time period: {(streaming_df['TimeStamp_UTC'].max() - streaming_df['TimeStamp_UTC'].min()).days} days")

print(f"\n🎵 Top Artist: {streaming_df['Performer'].value_counts().index[0]}")
print(f"  • Plays: {streaming_df['Performer'].value_counts().values[0]}")

print(f"\n🎶 Top Song: {streaming_df['Song'].value_counts().index[0]}")
print(f"  • Plays: {streaming_df['Song'].value_counts().values[0]}")

print(f"\n⏰ Listening Patterns:")
print(f"  • Peak hour: {streaming_df['Hour'].mode()[0]}:00")
print(f"  • Peak day: {day_names[streaming_df['DayOfWeek'].mode()[0]]}")
print(f"  • Average songs per day: {len(streaming_df) / (streaming_df['TimeStamp_UTC'].max() - streaming_df['TimeStamp_UTC'].min()).days:.1f}")

if len(features_available) > 0:
    print(f"\n🎼 Audio Features (Average):")
    for feature in audio_features[:5]:
        print(f"  • {feature.capitalize()}: {features_available[feature].mean():.3f}")

if len(ml_data) > 100:
    print(f"\n🤖 ML Model Performance:")
    print(f"  • Random Forest MAE: {mae:.2f}")
    print(f"  • Random Forest R²: {r2:.3f}")
    print(f"  • Neural Network Accuracy: {test_accuracy:.3f}")
    print(f"  • Neural Network AUC: {test_auc:.3f}")

print("\n" + "="*60)
print("Analysis completed successfully!")
print("="*60)